# Project 009: Random Forest Classifier & Ensemble Studio

**Category:** Machine Learning Basic Projects  
**Domain:** Foundational Supervised Learning & Bagging Ensembles  
**Dataset:** Wine Cultivars Chemical Recognition (178 samples, 13 features, 3 classes)  

---

### 📖 Theoretical Foundation

A **Random Forest** is an ensemble meta-estimator that fits $B$ independent, unpruned decision trees on bootstrap sub-samples of the dataset and aggregates their predictions by majority voting:

$$\hat{y} = \text{mode} \{ T_1(x), T_2(x), \dots, T_B(x) \}$$

#### ⚖️ The Bias-Variance Reduction Theorem in Bagging:
For $B$ identically distributed trees, each with variance $\sigma^2$ and average pairwise correlation $\rho$:

$$\text{Var}(\bar{T}) = \rho \sigma^2 + \frac{1 - \rho}{B} \sigma^2$$

- As $B \to \infty$, the second term decays to 0.
- By selecting $m \approx \sqrt{P}$ random features at every split, Random Forests decorrelate the trees, drastically shrinking $\rho$ and lowering total generalization error.

#### 🎲 Out-of-Bag (OOB) Error:
Since bootstrap sampling draws with replacement, $P(\text{OOB}) = (1 - 1/N)^N \approx 1/e \approx 36.8\%$ of observations are left out of each tree. This enables free, unbiased cross-validation during training.

---

### 🎯 Notebook Workflow
1. **Environment Setup & Imports**: Scikit-Learn, NumPy, Pandas, Matplotlib.
2. **Real Dataset Loading**: Wine Cultivars chemical profiling (178 samples, 13 features).
3. **Single Decision Tree vs Random Forest**: Demonstrating variance reduction through ensembling.
4. **Out-of-Bag (OOB) Error Convergence**: Asymptotic stabilization across tree counts.
5. **Mean Decrease in Impurity (Gini Feature Importance)**: Key chemical markers and ranking.
6. **Forest Structural Diagnostics**: Tree depth distribution and leaf complexity.
7. **Champion Model Evaluation**: Confusion Matrix, ROC-AUC, Precision, and Recall.
8. **2D Ensemble Decision Boundary & Voting Surface**: Visualizing non-linear partitions.

In [ ]:
# 1. Environment Setup & Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print("Scikit-learn and analytical libraries successfully loaded.")

In [ ]:
# 2. Real Dataset Loading: Wine Cultivars Chemical Recognition
raw_wine = load_wine()
df = pd.DataFrame(raw_wine.data, columns=raw_wine.feature_names)
df['target'] = raw_wine.target
target_names = [f"Cultivar {n}" for n in raw_wine.target_names]

print(f"Dataset Shape: {df.shape[0]} samples x {len(raw_wine.feature_names)} features")
print(f"Target Classes: {target_names}")
print(df['target'].value_counts())
df.head()

In [ ]:
# 3. Train/Test Stratified Split
X = df[raw_wine.feature_names].values
y = df['target'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f"Training Partition: {X_train.shape[0]} samples")
print(f"Holdout Test Partition: {X_test.shape[0]} samples")

In [ ]:
# 4. Single Decision Tree Baseline vs Random Forest Ensemble
single_tree = DecisionTreeClassifier(random_state=42).fit(X_train, y_train)
tree_acc = single_tree.score(X_test, y_test)

rf_model = RandomForestClassifier(n_estimators=100, max_features='sqrt', oob_score=True, random_state=42).fit(X_train, y_train)
rf_acc = rf_model.score(X_test, y_test)

print(f"Single Decision Tree Test Accuracy: {tree_acc:.2%}")
print(f"Random Forest (100 Trees) Test Accuracy: {rf_acc:.2%}")
print(f"Random Forest Out-of-Bag (OOB) Accuracy: {rf_model.oob_score_:.2%}")

In [ ]:
# 5. Out-of-Bag (OOB) Error Convergence Curve
n_trees_range = range(10, 160, 10)
oob_scores = []

rf_incremental = RandomForestClassifier(n_estimators=10, warm_start=True, oob_score=True, random_state=42)
for n in n_trees_range:
    rf_incremental.set_params(n_estimators=n)
    rf_incremental.fit(X_train, y_train)
    oob_scores.append(rf_incremental.oob_score_)

plt.figure(figsize=(7, 4))
plt.plot(list(n_trees_range), oob_scores, marker='o', color='#10b981', lw=2)
plt.title('Out-of-Bag (OOB) Accuracy vs Forest Size')
plt.xlabel('Number of Trees (n_estimators)')
plt.ylabel('OOB Accuracy')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# 6. Mean Decrease in Impurity (Gini Feature Importance Ranking)
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1]
sorted_features = [raw_wine.feature_names[i] for i in indices]

plt.figure(figsize=(8, 5))
plt.barh(range(len(indices)), importances[indices][::-1], color='#38bdf8', align='center')
plt.yticks(range(len(indices)), sorted_features[::-1])
plt.xlabel('Relative Feature Importance (Gini MDI)')
plt.title('Random Forest Feature Importance Ranking')
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
# 7. Comprehensive Model Evaluation on Test Partition
y_pred = rf_model.predict(X_test)
print("=== Classification Report on Unseen Holdout ===")
print(classification_report(y_test, y_pred, target_names=target_names))

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap='Greens')
for i in range(len(target_names)):
    for j in range(len(target_names)):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center', color='white' if cm[i, j] > cm.max()/2 else 'black', fontweight='bold')
ax.set_xticks(range(len(target_names)))
ax.set_yticks(range(len(target_names)))
ax.set_xticklabels([f'Pred {n}' for n in target_names], rotation=25)
ax.set_yticklabels([f'True {n}' for n in target_names])
ax.set_title("Confusion Matrix")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
# 8. 2D Ensemble Decision Boundary & Probability Surface
X_2d = X_train[:, [0, 9]] # Alcohol and Color Intensity
rf_2d = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42).fit(X_2d, y_train)

x_min, x_max = X_2d[:, 0].min() - 1, X_2d[:, 0].max() + 1
y_min, y_max = X_2d[:, 1].min() - 1, X_2d[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))

Z = rf_2d.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, Z, alpha=0.3, cmap='viridis')
scatter = plt.scatter(X_2d[:, 0], X_2d[:, 1], c=y_train, cmap='viridis', edgecolors='k')
plt.title('Random Forest 2D Decision Boundaries (Alcohol vs Color Intensity)')
plt.xlabel('Alcohol')
plt.ylabel('Color Intensity')
plt.legend(*scatter.legend_elements(), title='Cultivar')
plt.grid(True, alpha=0.25)
plt.show()